# BoM Hierarchy Explosion

Traverses the `material -> component` chain starting from every **FIN** material,
producing a flat list where each row is one link in the production hierarchy.

**Production stages:**  
High-purity polysilicon (8000) -> Single-crystal silicon ingot (8001) -> Lapped silicon wafer (8007) -> Polished silicon wafer (8002)

## 1. Load and inspect data

In [9]:
import pandas as pd
import numpy as np
from datetime import datetime

df = pd.read_csv("data/task_2_data_ex.csv")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

Shape: (1320, 11)
Columns: ['year', 'month', 'produced_material', 'produced_material_production_type', 'produced_material_release_type', 'produced_material_quantity', 'component_material', 'component_material_production_type', 'component_material_release_type', 'component_material_quantity', 'plant_id']


,year,month,produced_material,produced_material_production_type,produced_material_release_type,produced_material_quantity,component_material,component_material_production_type,component_material_release_type,component_material_quantity,plant_id
0,2024,1,10000,8002,FIN,990.00,50000,8002.0,PROD,990.00,RLT_10
1,2024,1,50000,8002,PROD,859.00,80070,8007.0,PROD,879.00,RLT_10
2,2024,1,50000,8002,PROD,859.00,90000,NaN,ADD,50.00,RLT_10
3,2024,1,50000,8002,PROD,859.00,90001,NaN,ADD,20.00,RLT_10
4,2024,1,80070,8007,PROD,929.00,80010,8001.0,PROD,"3,626.00",RLT_10


In [10]:
print("Release types:", df["produced_material_release_type"].value_counts().to_dict())
print("Plants:", list(df["plant_id"].unique()))
print("Years:", sorted(df["year"].unique()))
print("Months:", sorted(df["month"].unique()))

Release types: {'PROD': 1200, 'FIN': 120}
Plants: ['RLT_10', 'RLT_14', 'RLT_16']
Years: [np.int64(2024)]
Months: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12)]


## 2. Split FIN vs non-FIN rows

In [11]:
def split_by_release_type(df: pd.DataFrame):
    """Separate FIN (chain roots) from non-FIN (traversal candidates)."""
    fin  = df[df["produced_material_release_type"] == "FIN"].copy()
    prod = df[df["produced_material_release_type"] != "FIN"].copy()
    print(f"FIN rows (roots): {len(fin)}")
    print(f"Non-FIN rows (PROD/ADD/RM): {len(prod)}")
    return fin, prod

fin, prod = split_by_release_type(df)

FIN rows (roots): 120
Non-FIN rows (PROD/ADD/RM): 1200


## 3. BFS traversal

Starting from FIN materials, follow `component_material -> produced_material` links level by level.  
A `depth` counter tracks the hierarchy level so the output preserves the natural chain reading order.

In [12]:
def traverse_bom(fin: pd.DataFrame, prod: pd.DataFrame) -> pd.DataFrame:
    """
    BFS traversal of BoM hierarchy.
    
    - Starts from FIN rows (depth 0)
    - At each step: current component_material -> next produced_material
      (within same plant/year/month)
    - Carries fin_material_id through all levels
    - Tracks depth to preserve hierarchy reading order
    - Stops when no more children found
    """
    JOIN_KEYS = ["plant_id", "year", "month"]

    current = fin.copy()
    current["fin_material_id"] = current["produced_material"]
    current["depth"] = 0

    all_layers = []
    depth = 0

    while not current.empty:
        all_layers.append(current)
        print(f"  Depth {depth}: {len(current)} rows")

        lookup = current[
            JOIN_KEYS + ["component_material", "fin_material_id"]
        ].drop_duplicates()

        next_level = prod.merge(
            lookup,
            left_on  = JOIN_KEYS + ["produced_material"],
            right_on = JOIN_KEYS + ["component_material"],
            suffixes = ("", "_parent")
        )

        if "component_material_parent" in next_level.columns:
            next_level.drop(columns=["component_material_parent"], inplace=True)

        depth += 1
        next_level["depth"] = depth
        current = next_level

    result = pd.concat(all_layers, ignore_index=True)
    print(f"  Total: {len(result)} rows")
    return result

exploded = traverse_bom(fin, prod)

  Depth 0: 120 rows
  Depth 1: 360 rows
  Depth 2: 360 rows
  Depth 3: 240 rows
  Depth 4: 240 rows
  Total: 1320 rows


## 4. Format output

Sort by depth to keep the hierarchy readable top-to-bottom.

In [ ]:
def format_output(exploded: pd.DataFrame) -> pd.DataFrame:
    """Select, rename, and sort by depth to preserve chain reading order."""
    out = exploded[[
        "plant_id", "year", "month",
        "fin_material_id", "depth",
        "produced_material",
        "produced_material_release_type",
        "produced_material_production_type",
        "produced_material_quantity",
        "component_material",
        "component_material_release_type",
        "component_material_production_type",
        "component_material_quantity",
    ]].copy()

    out.columns = [
        "plant", "year", "month",
        "fin_material_id", "depth",
        "material",
        "release_type",
        "production_type",
        "production_quantity",
        "component",
        "component_release_type",
        "component_production_type",
        "component_quantity",
    ]

    return out.sort_values(
    ["plant", "year", "month", "fin_material_id", "depth",
     "component_release_type", "component"],
    ascending=[True, True, True, True, True, False, True]
).reset_index(drop=True)

result = format_output(exploded)
print(f"Output rows: {len(result)}")
result.head(20)

Output rows: 1320


,plant,year,month,fin_material_id,depth,material,release_type,production_type,production_quantity,component,component_release_type,component_production_type,component_quantity
0,RLT_10,2024,1,10000,0,10000,FIN,8002,990.00,50000,PROD,8002.0,990.00
1,RLT_10,2024,1,10000,1,50000,PROD,8002,859.00,80070,PROD,8007.0,879.00
2,RLT_10,2024,1,10000,1,50000,PROD,8002,859.00,90000,ADD,NaN,50.00
3,RLT_10,2024,1,10000,1,50000,PROD,8002,859.00,90001,ADD,NaN,20.00
4,RLT_10,2024,1,10000,2,80070,PROD,8007,929.00,80010,PROD,8001.0,"3,626.00"
5,RLT_10,2024,1,10000,2,80070,PROD,8007,929.00,90002,ADD,NaN,30.00
6,RLT_10,2024,1,10000,2,80070,PROD,8007,929.00,90003,ADD,NaN,11.00
7,RLT_10,2024,1,10000,3,80010,PROD,8001,"1,726.00",80000,PROD,8000.0,"1,818.00"
8,RLT_10,2024,1,10000,3,80010,PROD,8001,"1,726.00",90004,ADD,NaN,101.00
9,RLT_10,2024,1,10000,4,80000,PROD,8000,"1,980.00",70000,RM,NaN,"2,668.00"


## 5. Verify a single chain

Read one FIN material's chain top-to-bottom.

In [14]:
sample_fin   = result["fin_material_id"].iloc[0]
sample_month = result["month"].iloc[0]
sample_plant = result["plant"].iloc[0]

chain = result[
    (result["fin_material_id"] == sample_fin) &
    (result["month"] == sample_month) &
    (result["plant"] == sample_plant)
]

print(f"Chain for FIN {sample_fin} | Plant {sample_plant} | Month {sample_month}")
print(f"Rows: {len(chain)}")
chain[["plant", "year", "fin_material_id", "depth", "material", "release_type",
       "production_type", "component", "component_release_type"]]

Chain for FIN 10000 | Plant RLT_10 | Month 1
Rows: 11


,plant,year,fin_material_id,depth,material,release_type,production_type,component,component_release_type
0,RLT_10,2024,10000,0,10000,FIN,8002,50000,PROD
1,RLT_10,2024,10000,1,50000,PROD,8002,80070,PROD
2,RLT_10,2024,10000,1,50000,PROD,8002,90000,ADD
3,RLT_10,2024,10000,1,50000,PROD,8002,90001,ADD
4,RLT_10,2024,10000,2,80070,PROD,8007,80010,PROD
5,RLT_10,2024,10000,2,80070,PROD,8007,90002,ADD
6,RLT_10,2024,10000,2,80070,PROD,8007,90003,ADD
7,RLT_10,2024,10000,3,80010,PROD,8001,80000,PROD
8,RLT_10,2024,10000,3,80010,PROD,8001,90004,ADD
9,RLT_10,2024,10000,4,80000,PROD,8000,70000,RM


## 6. Annual summary

In [15]:
for year in sorted(result["year"].unique()):
    yr = result[result["year"] == year]
    n_fin = yr["fin_material_id"].nunique()
    n_plants = yr["plant"].nunique()
    print(f"Year {year}: {len(yr)} rows | {n_fin} FIN materials | {n_plants} plants")

Year 2024: 1320 rows | 10 FIN materials | 3 plants


## 7. Export

In [16]:
ts = datetime.today().strftime('%Y-%m-%d_%H-%M-%S')
filename = f'bom_explosion_{ts}.csv'
result.to_csv(filename, index=False)
print(f'Saved {len(result)} rows to {filename}')

Saved 1320 rows to bom_explosion_2026-03-03_14-28-47.csv
